In [55]:
import torch
import plotly.express as px
import pandas as pd
import numpy as np
from typing import Optional, List, Tuple
import pathlib
from torch import device
from epsilon_transformers import persistence
from epsilon_transformers.process.processes import PROCESS_REGISTRY

from pandas.core.computation.ops import Op
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
import pandas as pd

In [56]:
checkpoint_dir = pathlib.Path("/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/2_layer_0.05_0.85_lr1e-2_1b_ckpt50k/epsilon-transformers/models/mess3/2_layer_0.05_0.85_lr1e-2_1b_ckpt50k/")
if torch.cuda.is_available():
    device = device("cuda:0")
elif torch.backends.mps.is_available():
    device = device("mps")
else:
    device = device("cpu")

persister = Persister(checkpoint_dir)
model=persister.load_final_model(device=device)
train_config = persister.load_training_config()
#model=persister.load_model("path")
model.eval()

[Persister] Found 2004 checkpoints in /Users/sbhandari/Documents/GitHub/epsilon-transformers/models/2_layer_0.05_0.85_lr1e-2_1b_ckpt50k/epsilon-transformers/models/mess3/2_layer_0.05_0.85_lr1e-2_1b_ckpt50k


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (hook_resi

In [57]:
process_name = 'Mess3'
process_params ={
    "x": 0.05,
    "a": 0.85
}
seq_len = 10
vocab = 3
if process_name in PROCESS_REGISTRY:
    process=PROCESS_REGISTRY[process_name](**process_params)

In [5]:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)
print(input_seq)


tensor([[2, 2, 1, 2, 2, 2, 0, 2, 2, 2]], device='mps:0')


In [6]:
def get_attention_hooks(model):
    return [name for name,_ in model.named_modeules() if "attn.hook_pattern" in name]

In [ ]:
def plot_attention_heatmap(
    cache,
    layer_idx:int,
    head_idx: Optional[int]=None,
    tokens:Optional[list[str]]=None
):
    hook_name=f"blocks.{layer_idx}.attn.hook_pattern"
    if hook_name not in cache:
        print("error: hook not found in cache")
        return
    #pattern has shape [batch, n_heads, query_pos, key_pos]    
    pattern=cache[hook_name][0].detach().cpu().numpy()
    n_heads=pattern.shape[0]
    seq_len=pattern.shape[1]  

    if tokens is None:
        x_labels=[f"Key{i}" for i in range(seq_len)] 
        y_labels=[f"Query{i}" for i in range(seq_len)]
    else:
        x_labels = [f"{t} (K{i})" for i, t in enumerate(tokens)]
        y_labels = [f"{t} (Q{i})" for i, t in enumerate(tokens)]

    if head_idx is not None:
        print(f"Plotting Layer {layer_idx}, Head {head_idx}")
        fig = px.imshow(
            pattern[head_idx],
            labels=dict(x="Key (Source)", y="Query (Destination)", color="Attention"),
            x=x_labels,
            y=y_labels,
            title=f"Attention Pattern: Layer {layer_idx}, Head {head_idx}",
            color_continuous_scale="Viridis",
            range_color=[0, 1]
        )
        fig.update_layout(width=600, height=600)
    else:
        print(f"Plotting All Heads for Layer {layer_idx}")
        fig = px.imshow(
            pattern,
            labels=dict(x="Key", y="Query", color="Attn", facet_col="Head"),
            x=x_labels,
            y=y_labels,
            facet_col=0, # Facet over the first dimension (heads)
            facet_col_wrap=min(n_heads, 4), # Wrap after 4 heads
            title=f"Attention Patterns: Layer {layer_idx} (All Heads)",
            color_continuous_scale="Viridis",
            range_color=[0, 1]
        )
        fig.update_layout(height=400 * ((n_heads + 3) // 4), width=1200)

    fig.show()

In [58]:
logits, cache = model.run_with_cache(input_seq)

In [59]:
token_labels = [str(t.item()) for t in input_seq[0]]

In [60]:
plot_attention_heatmap(cache, layer_idx=0, head_idx=None, tokens=token_labels)

Plotting All Heads for Layer 0


In [61]:
token_colors = {
    0: '#1f77b4',  # Blue
    1: '#ff7f0e',  # Orange
    2: '#2ca02c'   # Green
}

In [62]:
def extract_vectors(model,layer_idx, head_idx):
    W_E=model.W_E.detach().cpu()
    W_V=model.blocks[layer_idx].attn.W_V[head_idx].detach().cpu()
    W_O=model.blocks[layer_idx].attn.W_O[head_idx].detach().cpu()
    if W_V.dim() > 2: W_V = W_V.squeeze()
    if W_O.dim() > 2: W_O = W_O.squeeze()
    print(W_E.shape, W_V.shape, W_O.shape)
    W_OV=W_V@W_O
    OV_vectors=W_E@W_OV
    return W_E, OV_vectors

In [ ]:
# def plot_pca_vectors(W_E, OV_vectors):
#     combined_data = torch.cat([W_E, OV_vectors], dim=0)
    
#     # Fit PCA
#     pca = PCA(n_components=2)
#     pca_result = pca.fit_transform(combined_data)
    
#     # Split back
#     we_pca = pca_result[:vocab]
#     ov_pca = pca_result[vocab:]
    
#     fig = go.Figure()
    
#     # Plot Embeddings 
#     for i in range(vocab):
#         fig.add_trace(go.Scatter(
#             x=[we_pca[i, 0]], y=[we_pca[i, 1]],
#             mode='markers',
#             marker=dict(size=10, color=token_colors[i], symbol='circle'),
#             name=f'Embedding {i}'
#         ))
        
#     # Plot OV Vectors
#     for i in range(vocab):
#         fig.add_trace(go.Scatter(
#             x=[0, ov_pca[i, 0]], 
#             y=[0, ov_pca[i, 1]],
#             mode='lines+markers',
#             marker=dict(size=8, color=token_colors[i], symbol='arrow-bar-up', angleref="previous"),
#             line=dict(width=3, color=token_colors[i]),
#             name=f'OV Vector {i}'
#         ))
        
#     fig.update_layout(
#         title="Figure 2A: PCA of Embeddings and OV Vectors",
#         xaxis_title="1st Principal Component",
#         yaxis_title="2nd Principal Component",
#         width=600, height=600,
#         template="plotly_white",
#         showlegend=True
#     )
#     fig.show()


In [79]:
# W_E, OV_vectors = extract_vectors(model,layer_idx=0, head_idx=1)
# plot_pca_vectors(W_E, OV_vectors)

In [50]:
def plot_attention_decay(model,num_seqs,layer_idx,head_idx):
    device=torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    if hasattr(process,'generate_batch_gpu'):
        input_batch=process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len,device=device)
    _,cache=model.run_with_cache(input_batch)
    patterns = cache[f"blocks.{layer_idx}.attn.hook_pattern"][:, head_idx, :, :].detach().cpu()
    indices = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
    valid_mask = indices <= 0
    batch_indices = indices.unsqueeze(0).expand(num_seqs, -1, -1)
    valid_distances = batch_indices[valid_mask.unsqueeze(0).expand(num_seqs, -1, -1)].flatten()
    valid_attentions = patterns[:, valid_mask].flatten()
    df_attn = pd.DataFrame({
        'distance': valid_distances.numpy(),
        'attention': valid_attentions.numpy()
    })
    
    # Group by distance and compute mean
    df_mean = df_attn.groupby('distance')['attention'].mean().reset_index()
    fig = go.Figure()
    
    # Scatter of all points (optional, can be messy) or just mean line
    fig.add_trace(go.Scatter(
        x=df_mean['distance'],
        y=df_mean['attention'],
        mode='lines+markers',
        name='Actual Attention',
        line=dict(color='black', width=2),
    ))
    
    fig.update_layout(
        title="Figure 2C: Attention Pattern vs Relative Source Position",
        xaxis_title="Source Position Relative to Destination",
        yaxis_title="Attention Value",
        template="plotly_white",
        width=800, height=600
    )
    fig.show()


In [51]:
plot_attention_decay(model, num_seqs=50, layer_idx=0, head_idx=1)

In [ ]:
def plot_real_ov_vectors(model, process, num_seqs=50, seq_len=10,layer_idx=0, head_idx=0):
    # 1. Generate Batch Data (Vectorized)
    if hasattr(process, 'generate_batch_gpu'):
        input_batch = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)
    # else:
    #     seqs = [process.generate_process_history(total_length=seq_len).symbols for _ in range(num_seqs)]
    #     input_batch = torch.tensor(seqs, dtype=torch.long, device=device)
        
    _, cache = model.run_with_cache(input_batch)
    
    #[Batch, seq_len, n_heads, d_head]
    v_batch = cache[f"blocks.{layer_idx}.attn.hook_v"][:, :, head_idx, :].cpu()

    W_O = model.blocks[layer_idx].attn.W_O[head_idx].detach().cpu() 
    if W_O.dim() > 2: W_O = W_O.squeeze()

    ov_updates_batch = v_batch @ W_O
    
    #[Batch * seq_len, d_model]
    all_vectors = ov_updates_batch.reshape(-1, ov_updates_batch.shape[-1])
    #flatten labels
    labels = input_batch.cpu().flatten().tolist()

    #token embedding
    static_embeddings = model.W_E.detach().cpu()

    #pca
    pca = PCA(n_components=2)
    pca.fit(all_vectors)
    
    ov_pca = pca.transform(all_vectors)
    emb_pca = pca.transform(static_embeddings)
    fig = go.Figure()
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for i in range(model.cfg.d_vocab):
        fig.add_trace(go.Scatter(
            x=[emb_pca[i, 0]], y=[emb_pca[i, 1]],
            mode='markers',
            marker=dict(size=10, color=colors[i], symbol='circle', line=dict(width=1, color='black')),
            name=f'Embedding {i}'
        ))
    
    for token_id in range(model.cfg.d_vocab):
        indices = [idx for idx, label in enumerate(labels) if label == token_id]
        if not indices: continue
            
        #mean ov vector for the token
        token_ov_vectors = ov_pca[indices]
        mean_vector = token_ov_vectors.mean(axis=0)
        
        # arrow from origin to mean vector
        fig.add_trace(go.Scatter(
            x=[0, mean_vector[0]], 
            y=[0, mean_vector[1]],
            mode='lines+markers',
            marker=dict(size=12, symbol='arrow-bar-up', angleref="previous", color=colors[token_id]),
            line=dict(width=4, color=colors[token_id]),
            name=f'OV Vector {token_id}'
        ))
        fig.add_trace(go.Scatter(
            x=token_ov_vectors[:, 0], 
            y=token_ov_vectors[:, 1], 
            mode='markers',
            marker=dict(size=4, color=colors[token_id], opacity=0.3),
            showlegend=False
        ))

    fig.update_layout(
        title=f" Embeddings vs OV Vectors (Layer {layer_idx}, Head {head_idx})",
        xaxis_title="PC1", yaxis_title="PC2",
        width=800, height=800,
        template="plotly_white",
        yaxis=dict(scaleanchor="x", scaleratio=1)
    )
    fig.show()


In [78]:
plot_real_ov_vectors(model, process, num_seqs=10, seq_len=10, layer_idx=0, head_idx=1)